# Celligner expression alignment

Align TCGA tumor and DepMap cell-line expression profiles.

## Dependencies

Install the Broad Institute Celligner development version with its bundled mnnpy submodule before running the alignment.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

project_root = os.environ.get("MSS_TMB_PROJECT_ROOT")
if project_root:
    PROJECT_ROOT = Path(project_root).expanduser().resolve()
else:
    cwd = Path.cwd()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "scripts").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Set MSS_TMB_PROJECT_ROOT to the project directory.")


import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import softmax
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu, spearmanr

from sklearn.decomposition import PCA
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## Configuration

In [ ]:
DATA_DIR = (PROJECT_ROOT / "data")
OUTPUT_DIR = DATA_DIR / "celligner_mss_tmbh_screen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEPMAP_EXPRESSION_PATH = DATA_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
DEPMAP_MODEL_PATH = DATA_DIR / "Model.csv"
PRISM_RESPONSE_PATH = DATA_DIR / "secondary-screen-dose-response-curve-parameters.csv"
PRISM_TREATMENT_INFO_PATH = DATA_DIR / "secondary-screen-replicate-collapsed-treatment-info.csv"

# Expression may be sample x gene or gene x sample; orientation is detected using metadata IDs.

TUMOR_DATASETS = {
    "CRC": {
        "expression_path": next((DATA_DIR / f for f in ("crc_tcga_log2_tpm.csv", "CRC_TCGA_LOG2TPM.csv") if (DATA_DIR / f).is_file()), DATA_DIR / "crc_tcga_log2_tpm.csv"),
        "metadata_path": DATA_DIR / "clinical_data.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
    "UCEC": {
        "expression_path": next((DATA_DIR / f for f in ("ucec_tcga_log2_tpm.csv", "UCEC_TCGA_LOG2TPM.csv") if (DATA_DIR / f).is_file()), DATA_DIR / "ucec_tcga_log2_tpm.csv"),
        "metadata_path": DATA_DIR / "ucec_clinical.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
}

EXTRA_ALIGNMENT_DATASETS = [

]

CELLIGNER_MODE = "run"
CELLIGNER_ALIGNED_PATH = OUTPUT_DIR / "celligner_aligned_expression.pkl"
CELLIGNER_MODEL_PATH = OUTPUT_DIR / "celligner_model.pkl"
CELLIGNER_LOW_MEMORY = True

CLASS_ORDER = ["MSS/TMB-L", "MSS/TMB-H", "MSI-H"]
K_CANDIDATES = [1, 3, 5, 7]
TEMPERATURE_CANDIDATES = [0.03, 0.05, 0.10, 0.20, 0.50]
PCA_VARIANCE = 0.90
MAX_VARIABLE_FEATURES = 5000
CV_REPEATS = 10

PRIMARY_PROBABILITY_THRESHOLD = 0.60
PRIMARY_MARGIN_THRESHOLD = 0.10
THRESHOLDS_TO_TEST = [0.50, 0.60, 0.70]

MIN_MODELS_PER_GROUP = 3
MIN_MODELS_CONTINUOUS = 10
FDR_CUTOFF = 0.05

CRC_ONCOTREE_CODES = ["COAD", "READ", "MACR", "COADREAD"]
UTERINE_CARCINOMA_CODES = ["UCEC", "UCS", "UCCC", "UASC"]
UTERINE_BROAD_CODES = ["UCEC", "UCS", "UCCC", "UASC", "USARC", "ESS", "ULMS"]

## Harmonize input identifiers

In [ ]:
def read_table(path: Path, index_col: Optional[int] = 0) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    suffix = "".join(path.suffixes).lower()
    if suffix.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffix.endswith((".pkl", ".pickle")):
        return pd.read_pickle(path)
    if suffix.endswith((".tsv", ".txt")):
        return pd.read_csv(path, sep="\t", index_col=index_col)
    return pd.read_csv(path, index_col=index_col)

def canonical_sample_id(value: object) -> str:
    value = str(value).strip().replace(".", "-")
    if value.upper().startswith("TCGA-") and len(value) >= 16:
        value = value[:16]
    return value

def clean_gene_id(value: object) -> str:
    value = str(value).strip()
    if " (" in value:
        value = value.split(" (", 1)[0]
    if "|" in value:
        value = value.split("|", 1)[0]
    if value.upper().startswith("ENSG") and "." in value:
        value = value.split(".", 1)[0]
    return value.upper()

def collapse_duplicate_genes(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x.columns = [clean_gene_id(c) for c in x.columns]
    return x.T.groupby(level=0).mean().T

def normalize_group_label(value: object):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().upper().replace("_", "-").replace(" ", "")
    mapping = {
        "MSS/TMB-L": "MSS/TMB-L", "MSS-TMB-L": "MSS/TMB-L", "MSS/TMBL": "MSS/TMB-L",
        "MSS/TMB-H": "MSS/TMB-H", "MSS-TMB-H": "MSS/TMB-H", "MSS/TMBH": "MSS/TMB-H",
        "MSI-H": "MSI-H", "MSIH": "MSI-H", "MSI/H": "MSI-H",
    }
    return mapping.get(key, str(value).strip())

def load_metadata(path, cancer, group_col, sample_id_col=None):
    meta = read_table(path, index_col=0 if sample_id_col is None else None)
    ids = pd.Series(meta.index, index=meta.index) if sample_id_col is None else meta[sample_id_col]
    if group_col not in meta.columns:
        raise KeyError(f"{group_col!r} not found in {path}. Columns: {list(meta.columns)}")
    out = pd.DataFrame({
        "sample_id": ids.map(canonical_sample_id).values,
        "cancer": cancer,
        "group": meta[group_col].map(normalize_group_label).values,
    })
    out = out[out["group"].isin(CLASS_ORDER)].drop_duplicates("sample_id")
    out["aligned_id"] = "TCGA::" + out["sample_id"]
    return out.set_index("sample_id", drop=False)

def orient_to_samples(x: pd.DataFrame, expected_ids=None):
    x = x.copy()
    x.index = x.index.map(canonical_sample_id)
    x.columns = [canonical_sample_id(c) for c in x.columns]
    if expected_ids is None:
        return x
    expected = {canonical_sample_id(i) for i in expected_ids}
    row_overlap = len(expected.intersection(x.index))
    col_overlap = len(expected.intersection(x.columns))
    if col_overlap > row_overlap:
        x = x.T
        x.index = x.index.map(canonical_sample_id)
    if max(row_overlap, col_overlap) == 0:
        warnings.warn("No metadata IDs matched either expression axis; assuming samples are rows.")
    return x

def load_tumor_expression(path, metadata_ids, already_log2_tpm):
    x = orient_to_samples(read_table(path, index_col=0), metadata_ids)
    x = x.apply(pd.to_numeric, errors="coerce")
    if not already_log2_tpm:
        x = np.log2(x.clip(lower=0) + 1)
    x = collapse_duplicate_genes(x)
    return x.groupby(level=0).mean()

def load_depmap_expression(path):
    x = read_table(path, index_col=0)
    if "MODELID" in x.columns:
        x = x.set_index("MODELID")
    elif "ModelID" in x.columns:
        x = x.set_index("ModelID")
    x.index = x.index.astype(str)
    x = x.drop(columns=["MODELID", "ModelID"], errors="ignore")
    x = x.apply(pd.to_numeric, errors="coerce")
    return collapse_duplicate_genes(x).groupby(level=0).mean()

## Load data

In [ ]:
depmap_expression = load_depmap_expression(DEPMAP_EXPRESSION_PATH)
models = read_table(DEPMAP_MODEL_PATH, index_col=None)
prism = read_table(PRISM_RESPONSE_PATH, index_col=None)
treatment_info = read_table(PRISM_TREATMENT_INFO_PATH, index_col=None)

required_model_columns = {"ModelID", "OncotreeCode"}
missing = required_model_columns.difference(models.columns)
if missing:
    raise KeyError(f"Missing Model.csv columns: {sorted(missing)}")

tumor_expression_parts = []
tumor_metadata_parts = []

for cancer, config in TUMOR_DATASETS.items():
    meta = load_metadata(
        config["metadata_path"], cancer, config["group_col"], config["sample_id_col"]
    )
    expr = load_tumor_expression(
        config["expression_path"], meta["sample_id"], config["already_log2_tpm"]
    )
    common_ids = meta.index.intersection(expr.index)
    if len(common_ids) == 0:
        raise ValueError(f"No matched metadata/expression samples for {cancer}.")
    meta = meta.loc[common_ids].copy()
    expr = expr.loc[common_ids].copy()
    expr.index = "TCGA::" + expr.index
    tumor_metadata_parts.append(meta)
    tumor_expression_parts.append(expr)

tumor_metadata = pd.concat(tumor_metadata_parts)
labeled_tumor_expression = pd.concat(tumor_expression_parts, axis=0, join="outer")

extra_parts = []
for config in EXTRA_ALIGNMENT_DATASETS:
    extra = load_tumor_expression(
        config["expression_path"], None, config["already_log2_tpm"]
    )
    extra.index = "EXTRA::" + str(config["name"]) + "::" + extra.index.astype(str)
    extra_parts.append(extra)

all_tumor_expression = pd.concat(
    [labeled_tumor_expression] + extra_parts, axis=0, join="outer"
)

print("DepMap:", depmap_expression.shape)
print("Labeled tumors:", labeled_tumor_expression.shape)
print("All alignment tumors:", all_tumor_expression.shape)
display(tumor_metadata.groupby(["cancer", "group"]).size().rename("n").reset_index())

## Align expression profiles

In [ ]:
common_genes = depmap_expression.columns.intersection(all_tumor_expression.columns)
if len(common_genes) < 1000:
    raise ValueError(
        f"Only {len(common_genes)} overlapping genes. Harmonize gene identifiers first."
    )

ccle_input = depmap_expression.loc[:, common_genes].copy()
ccle_input.index = "CL::" + ccle_input.index.astype(str)
tumor_input = all_tumor_expression.loc[:, common_genes].copy()

ccle_input = ccle_input.fillna(ccle_input.median())
tumor_input = tumor_input.fillna(tumor_input.median())
usable = (
    ccle_input.notna().all() & tumor_input.notna().all()
    & ((ccle_input.var() > 0) | (tumor_input.var() > 0))
)
ccle_input = ccle_input.loc[:, usable]
tumor_input = tumor_input.loc[:, usable]

expected_ids = set(ccle_input.index).union(tumor_input.index)

def normalize_aligned_output(x):
    x = pd.DataFrame(x).copy()
    row_overlap = len(expected_ids.intersection(x.index.astype(str)))
    col_overlap = len(expected_ids.intersection(x.columns.astype(str)))
    if col_overlap > row_overlap:
        x = x.T
    x.index = x.index.astype(str)
    return x.apply(pd.to_numeric, errors="coerce")

if CELLIGNER_MODE == "run":
    try:
        from celligner import Celligner
    except ImportError as exc:
        raise ImportError(
            "Celligner is not installed. Install it or set CELLIGNER_MODE='load'."
        ) from exc
    try:
        aligner = Celligner(low_memory=CELLIGNER_LOW_MEMORY)
    except TypeError:
        aligner = Celligner()
        if hasattr(aligner, "low_memory"):
            aligner.low_memory = CELLIGNER_LOW_MEMORY
    aligner.fit(ccle_input)
    aligner.transform(tumor_input)
    aligned_expression = normalize_aligned_output(aligner.combined_output)
    aligned_expression.to_pickle(CELLIGNER_ALIGNED_PATH)
    try:
        aligner.save(str(CELLIGNER_MODEL_PATH))
    except Exception as exc:
        warnings.warn(f"Could not save Celligner model: {exc}")
elif CELLIGNER_MODE == "load":
    aligned_expression = normalize_aligned_output(pd.read_pickle(CELLIGNER_ALIGNED_PATH))
else:
    raise ValueError("CELLIGNER_MODE must be 'run' or 'load'.")

print("Aligned matrix:", aligned_expression.shape)
print("Expected samples missing:", len(expected_ids.difference(aligned_expression.index)))

## Inspect alignment with PCA

In [ ]:
def plot_source_pca(matrix, max_features=3000):
    source = pd.Series(index=matrix.index, dtype=object)
    source.loc[source.index.str.startswith("CL::")] = "Cell line"
    source.loc[source.index.str.startswith("TCGA::")] = "TCGA tumor"
    source.loc[source.index.str.startswith("EXTRA::")] = "Other tumor"

    ids = source.dropna().index.intersection(matrix.index)
    x = matrix.loc[ids].copy()
    features = x.var().sort_values(ascending=False).head(max_features).index
    x = x[features].fillna(x[features].median())
    coordinates = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(x)
    )

    plt.figure(figsize=(8, 6))
    for label in source.loc[ids].unique():
        mask = source.loc[ids].values == label
        plt.scatter(coordinates[mask, 0], coordinates[mask, 1], s=18, alpha=0.65, label=label)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Celligner-aligned expression")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

plot_source_pca(aligned_expression)